**`prepare_cheer_ground_truth`**

Transforms the CHEER summer-scholars occupancy survey (an Excel workbook of student-collected building photos, coordinates, and hand-classified occupancy types across several NC counties) into a single cleaned CSV, then tests linking it against the curated `US_footprint-openplaces-2026` footprint inventory to serve as a ground-truth test set for `occupancy_type`.

## Source data

`US-NC_footprint-cheer-summerscholars.xlsx` has one tab per student, each covering one primary NC county. Rows carry a `Street Address`/`Town` pair (reference-only -- the workbook's own instructions say students could survey any property, not just the listed ones), a `Photo File Name` encoding an occupancy-type prefix and the pinned lat/lon (e.g. `MFH_35.5389, -77.0509`), and a `Classification (SFH, MFH, MMH)` column.

Manual inspection turned up three things worth handling explicitly rather than trusting the sheet structure at face value:

- The last tab (`Chris+Daniel - Robeson & others`) is almost entirely `Street Address`/`Town`-empty: only lat/lon + classification were recorded for most of its rows, and one row is a straight copy-paste of a coordinate from the Beaufort tab. The declared per-tab county cannot be trusted.
- A `Photo File Name` prefix occasionally disagrees with the `Classification` column, and a few `Classification` values are free-text notes ('street view/info unavailable', 'blocked by a huge tree', ...) rather than a class.
- Some `Street Address` values are already full one-line addresses (`105 Village Cir, Washington, NC 27889`) while most are bare street lines -- naively appending `Town, NC` to every row double-counts the town/state on the former.

Given this, the pinned lat/lon is treated as the one authoritative field; everything else (address, town, county) is cleaned/derived but not trusted blindly.

In [ ]:
import re

import geopandas as gpd
import pandas as pd

import openplaces as op
from openplaces.geo.address import parse_address

XLSX_PATH = (
    op.cfg.external_dir
    / 'US'
    / 'NC'
    / '_all'
    / 'footprint'
    / 'cheer'
    / 'summerscholars2026'
    / 'US-NC_footprint-cheer-summerscholars.xlsx'
)
OUT_CSV = XLSX_PATH.with_suffix('.csv')

## Parse sheets

Each per-student tab has the same layout once the two metadata rows above the real header are skipped: row 0 is `Student Name: .../County: ...`, row 1 is blank ('You only really need to fill out this column'), row 2 is the real header (`Street Address`, `Town`, `Photo File Name`, `Classification (SFH, MFH, MMH):`), and data starts at row 3.

`parse_latlon`/`parse_prefix` pull the coordinate pair and the leading type code out of `Photo File Name` (format varies: `MFH_lat, lon`, `MMHlat, lon` with no separator, or occasionally a stray digit prefix). `normalize_class` accepts the `Classification` column only when it resolves unambiguously to exactly one of `SFH`/`MFH`/`MMH` (so 'MFH (apartments)' still counts, but 'MFH or SFH' and free-text notes are correctly rejected).

In [ ]:
_LATLON_RE = re.compile(r'(-?\d{1,3}\.\d+)\s*,\s*(-?\d{1,3}\.\d+)')
_PREFIX_RE = re.compile(r'^([A-Za-z]+)')
_VALID_CODES = {'SFH', 'MFH', 'MMH'}
_CANONICAL_MAP = {
    'SFH': 'Single-Family',
    'MFH': 'Multi-Family',
    'MMH': 'Manufactured Home',
}


def parse_latlon(photo_name):
    if not isinstance(photo_name, str):
        return None, None
    m = _LATLON_RE.search(photo_name)
    if not m:
        return None, None
    return float(m.group(1)), float(m.group(2))


def parse_prefix(photo_name):
    if not isinstance(photo_name, str):
        return None
    m = _PREFIX_RE.match(photo_name.strip())
    if not m:
        return None
    code = m.group(1).upper()
    return code if code in _VALID_CODES else None


def normalize_class(raw):
    if not isinstance(raw, str):
        return None
    s = raw.strip().upper()
    if s in _VALID_CODES:
        return s
    found = set(re.findall(r'\b(SFH|MFH|MMH)\b', s))
    return found.pop() if len(found) == 1 else None


def load_sheet(xl, sheet_name):
    """Read one student tab, skipping the two metadata rows above the header."""
    raw = xl.parse(sheet_name, header=None)
    header_row = raw.iloc[2]
    data = raw.iloc[3:].copy()
    data.columns = [str(c).strip() if isinstance(c, str) else c for c in header_row]
    data = data.reset_index(drop=True)
    data['source_sheet'] = sheet_name
    data['source_row'] = data.index + 4  # 1-based excel row of the data row
    return data

In [ ]:
xl = pd.ExcelFile(XLSX_PATH)
sheets = [s for s in xl.sheet_names if s != 'PLEASE READ Instructions + Note']

frames = []
for sheet in sheets:
    data = load_sheet(xl, sheet)
    street_col = next(
        (
            c
            for c in data.columns
            if isinstance(c, str) and c.startswith('Street Address')
        ),
        None,
    )
    town_col = next(
        (c for c in data.columns if isinstance(c, str) and c.startswith('Town')), None
    )
    photo_col = next(
        (c for c in data.columns if isinstance(c, str) and c.startswith('Photo')), None
    )
    class_col = next(
        (
            c
            for c in data.columns
            if isinstance(c, str) and c.startswith('Classification')
        ),
        None,
    )

    frames.append(
        pd.DataFrame(
            {
                'source_sheet': data['source_sheet'],
                'source_row': data['source_row'],
                'street_address_raw': data[street_col] if street_col else None,
                'town_raw': data[town_col] if town_col else None,
                'photo_file_name_raw': data[photo_col] if photo_col else None,
                'classification_raw': data[class_col] if class_col else None,
            }
        )
    )

combined = pd.concat(frames, ignore_index=True)

combined[['lat', 'lon']] = combined['photo_file_name_raw'].apply(
    lambda v: pd.Series(parse_latlon(v))
)
combined['photo_prefix'] = combined['photo_file_name_raw'].apply(parse_prefix)
combined['classification_norm'] = combined['classification_raw'].apply(normalize_class)

# The explicit Classification column is the student's deliberate final answer;
# the photo-prefix is only a fallback for the (few) rows where it's missing.
combined['occupancy_type'] = combined['classification_norm'].combine_first(
    combined['photo_prefix']
)
combined['occupancy_type_source'] = None
combined.loc[combined['classification_norm'].notna(), 'occupancy_type_source'] = (
    'classification_column'
)
combined.loc[
    combined['classification_norm'].isna() & combined['photo_prefix'].notna(),
    'occupancy_type_source',
] = 'photo_prefix'
combined['occupancy_type_conflict'] = (
    combined['classification_norm'].notna()
    & combined['photo_prefix'].notna()
    & (combined['classification_norm'] != combined['photo_prefix'])
)
combined['occupancy_type_canonical'] = combined['occupancy_type'].map(_CANONICAL_MAP)

# Usable for point-based linkage to a building inventory: needs both a
# geolocation and a resolved ground-truth class.
usable = combined[combined['lat'].notna() & combined['occupancy_type'].notna()].copy()

print(f'Raw rows across {len(sheets)} sheets: {len(combined)}')
print(f'Usable rows (parseable lat/lon + resolved occupancy_type): {len(usable)}')
print(
    f'Column/photo-prefix conflicts: {int(combined["occupancy_type_conflict"].sum())}'
)
combined['occupancy_type'].value_counts(dropna=False)

## Clean addresses and resolve county from coordinates

Addresses are parsed with `openplaces.geo.address.parse_address` (the same USPS-normalization backend the harmonizer uses). `clean_one_address` avoids re-appending `Town, NC` when `street_address_raw` already spells out a full address.

Two structured matching keys are also derived for `link_points_to_entities`:

- `address_number` and `address_street`, parsed from the raw street line.
- Parsed from the raw line rather than the cleaned string, because display normalization abbreviates suffixes (`ROAD` → `Rd`); `match_streets` canonicalizes both forms at comparison time.

The county (`admin_id`) is resolved by spatially joining each point against the NC county polygons rather than trusting the sheet's tab label -- necessary given the Robeson tab's copy-pasted Beaufort coordinate and the general unreliability of the reference `Town` column noted above.

In [ ]:
def clean_one_address(street, town):
    """Build a one-line address string, avoiding a duplicated town/state
    when street_address_raw already spells out the full address (a mix
    seen across sheets/students)."""
    street = street.strip() if isinstance(street, str) else ''
    town = town.strip() if isinstance(town, str) else ''
    if not street and not town:
        return None

    street_upper = street.upper()
    already_has_town = bool(town) and town.upper() in street_upper
    already_has_state_or_zip = bool(re.search(r'\bNC\b', street_upper)) or bool(
        re.search(r'\d{5}', street_upper)
    )

    if street and (already_has_town or already_has_state_or_zip or ',' in street):
        addr_str = street if already_has_state_or_zip else f'{street}, NC'
    elif street and town:
        addr_str = f'{street}, {town}, NC'
    elif street:
        addr_str = f'{street}, NC'
    else:
        addr_str = f'{town}, NC'

    return parse_address(addr_str, admin1_id='US')


parsed = usable.apply(
    lambda row: clean_one_address(row['street_address_raw'], row['town_raw']), axis=1
)
usable['address_cleaned'] = parsed.apply(lambda p: p.address_formatted if p else None)
usable['city'] = parsed.apply(lambda p: p.components.get('city') if p else None)
usable['state'] = parsed.apply(lambda p: p.components.get('state') if p else None)
usable['postal_code'] = parsed.apply(
    lambda p: p.components.get('postal_code') if p else None
)


def parse_number_street(street):
    """Matching keys from the raw street line, for link_points_to_entities."""
    if not isinstance(street, str) or not street.strip():
        return None, None
    components = parse_address(street, admin1_id='US').components
    return components.get('address_number'), components.get('address_street')


usable[['address_number', 'address_street']] = usable['street_address_raw'].apply(
    lambda s: pd.Series(parse_number_street(s))
)

counties = op.get_admin('US-NC', level=3, geom=True)
points = gpd.GeoDataFrame(
    usable, geometry=gpd.points_from_xy(usable['lon'], usable['lat']), crs='EPSG:4326'
)

right_index_col = counties.index.name or 'index_right'
joined = gpd.sjoin(
    points, counties[['name', 'geometry']], how='left', predicate='within'
)
joined = joined.rename(
    columns={'name': 'county_name', right_index_col: 'admin_id'}
).drop(columns=['geometry'])

print(f'Rows outside every NC county polygon: {int(joined["admin_id"].isna().sum())}')
joined.groupby(['admin_id', 'county_name'], dropna=False).size()

In [ ]:
FINAL_COLS = [
    'admin_id',
    'county_name',
    'occupancy_type',
    'occupancy_type_canonical',
    'occupancy_type_source',
    'occupancy_type_conflict',
    'lat',
    'lon',
    'address_cleaned',
    'address_number',
    'address_street',
    'city',
    'state',
    'postal_code',
    'street_address_raw',
    'town_raw',
    'source_sheet',
    'source_row',
    'photo_file_name_raw',
]
final = (
    joined[FINAL_COLS]
    .sort_values(['admin_id', 'source_sheet', 'source_row'])
    .reset_index(drop=True)
)
final.to_csv(OUT_CSV, index=False)
print(f'Wrote {len(final)} rows to {OUT_CSV}')
final.sample(5).T

## Test linkage against `US_footprint-openplaces-2026` outputs

Brunswick, Carteret, Dare, and Pender already had curated CHEER footprint output. Beaufort, Halifax, Johnston, and Robeson did not, so the pipeline was run for them (`ingest`/`harmonize`/`enrich`/`curate` via `scripts/examples/US_curate_footprints.py`, with `--include_streetview`/`--include_googlesatellite` left off -- no imagery enrichment, matching the other 4 counties' baseline). All 8 counties now have output.

A straight point-in-polygon join badly undercounts matches (as low as 33-38% in Brunswick/Dare/Pender) even though most points are only a few meters from their true footprint -- Street View 'right-click to get coordinates' pins commonly land just outside the roof polygon rather than inside it. `sjoin_nearest` with a 15m cutoff is used instead, which is comfortably larger than the observed pin offsets (Brunswick: median 1.7m, 98% under 20m) but still tight enough not to grab an unrelated neighboring building.

One caveat surfaced by the Robeson run: `link_curated_entity` matched only 154 of 119,155 footprint rows to `US_parcel-openplaces-2026` (vs. tens of thousands for the other new counties), so Robeson's occupancy classification leans much more heavily on the NSI/morphology/Overture fallbacks than on parcel evidence -- worth keeping in mind when reading its agreement numbers below.

In [ ]:
MAX_DIST_M = 15
_COLLAPSE_MULTI_FAMILY = {
    'Low-Rise Multi-Family': 'Multi-Family',
    'Mid-Rise Multi-Family': 'Multi-Family',
    'High-Rise Multi-Family': 'Multi-Family',
}
ALL_COUNTIES = [
    'US-NC-BR',
    'US-NC-AR',
    'US-NC-DR',
    'US-NC-PE',  # already had output
    'US-NC-BE',
    'US-NC-HL',
    'US-NC-JO',
    'US-NC-RB',  # generated above
]

for admin_id in ALL_COUNTIES:
    sub = final[final['admin_id'] == admin_id]
    footprints = op.get_entities('US_footprint-openplaces-2026', admin_id, geom=True)

    metric_crs = footprints.estimate_utm_crs()
    points = gpd.GeoDataFrame(
        sub, geometry=gpd.points_from_xy(sub['lon'], sub['lat']), crs='EPSG:4326'
    ).to_crs(metric_crs)
    footprints_m = footprints[['occupancy_type', 'geometry']].to_crs(metric_crs)

    nearest = gpd.sjoin_nearest(
        points,
        footprints_m,
        how='left',
        distance_col='dist_m',
        lsuffix='gt',
        rsuffix='inv',
    )
    nearest = (
        nearest.sort_values('dist_m').groupby(level=0).first()
    )  # keep closest on ties

    matched = nearest['dist_m'] <= MAX_DIST_M
    print(f'\n=== {admin_id} ({sub["county_name"].iloc[0]}) ===')
    print(
        f'Ground-truth points: {len(sub)}  |  Matched within {MAX_DIST_M}m: {matched.sum()} ({matched.mean():.1%})'
    )

    both = nearest[matched].copy()
    both['inv_occupancy_collapsed'] = both['occupancy_type_inv'].replace(
        _COLLAPSE_MULTI_FAMILY
    )
    agree = both['occupancy_type_canonical'] == both['inv_occupancy_collapsed']
    print(
        f'Occupancy agreement (Multi-Family bands collapsed): {agree.sum()}/{len(both)} ({agree.mean():.1%})'
    )
    print(
        pd.crosstab(
            both['occupancy_type_canonical'],
            both['inv_occupancy_collapsed'],
            rownames=['ground truth'],
            colnames=['inventory'],
        )
    )

## Summary

All 8 surveyed counties now have curated `US_footprint-openplaces-2026` output and were linkage-tested above. Occupancy agreement ranges widely (from ~42% in Pender to ~89% in Johnston) -- expected for a ground-truth test set whose purpose is to surface where the automated classifier is right or wrong, not a bug in the linkage itself. Robeson's low parcel-match rate (noted above) is the one result that should be read with caution rather than taken as a straightforward classifier accuracy signal.